# HDF5 Data Inspection Notebook

This notebook provides utilities to inspect the HDF5 data files generated by the `data_recorder.py` module.


In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np

# Configure matplotlib for notebook display
%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 8)

## Load HDF5 File


In [ ]:
# Path to the HDF5 file - update this to your file path
DATA_PATH = "/tmp/agile_recording/data.h5"

# Open the file with error handling
try:
    h5_file = h5py.File(DATA_PATH, "r")
    print(f"✓ Opened: {DATA_PATH}")
except OSError as e:
    if "bad object header" in str(e):
        print("✗ Error: File appears to be corrupted (likely not closed properly).")
        print("  This happens when the recording script is killed without proper cleanup.")
        print("  Make sure to use 'with recorder:' context manager or call recorder.close().")
    else:
        print(f"✗ Error opening file: {e}")
    raise

## File Metadata


In [ ]:
# Display file-level metadata
print("=== File Attributes ===")
for key, value in h5_file.attrs.items():
    print(f"  {key}: {value}")

# Display data group metadata
if "data" in h5_file:
    data_grp = h5_file["data"]
    print("\n=== Data Group Attributes ===")
    for key, value in data_grp.attrs.items():
        print(f"  {key}: {value}")

## File Structure


In [ ]:
def print_hdf5_structure(group, indent=0):
    """Recursively print HDF5 file structure."""
    prefix = "  " * indent
    for key in group.keys():
        item = group[key]
        if isinstance(item, h5py.Group):
            print(f"{prefix}📁 {key}/")
            # Only expand first few demos to avoid clutter
            if key.startswith("demo_") and int(key.split("_")[1]) > 2:
                print(f"{prefix}  ...")
                continue
            print_hdf5_structure(item, indent + 1)
        else:
            # It's a dataset
            print(f"{prefix}📊 {key}: shape={item.shape}, dtype={item.dtype}")


print("=== HDF5 Structure ===")
print_hdf5_structure(h5_file)

## Demo Statistics


In [ ]:
data_grp = h5_file["data"]
demo_names = [k for k in data_grp.keys() if k.startswith("demo_")]
num_demos = len(demo_names)

print(f"Number of demos: {num_demos}")

# Collect episode lengths
episode_lengths = []
for demo_name in demo_names:
    demo = data_grp[demo_name]
    num_samples = demo.attrs.get("num_samples", 0)
    episode_lengths.append(num_samples)

episode_lengths = np.array(episode_lengths)
print("\nEpisode length statistics:")
print(f"  Min: {episode_lengths.min()}")
print(f"  Max: {episode_lengths.max()}")
print(f"  Mean: {episode_lengths.mean():.2f}")
print(f"  Std: {episode_lengths.std():.2f}")
print(f"  Total samples: {episode_lengths.sum()}")

## Inspect Single Demo


In [ ]:
# Select a demo to inspect
DEMO_IDX = 0
demo_name = f"demo_{DEMO_IDX}"
demo = data_grp[demo_name]

print(f"=== {demo_name} ===")
print("Attributes:")
for key, value in demo.attrs.items():
    print(f"  {key}: {value}")

print("\nDatasets:")


def list_datasets(group, prefix=""):
    for key in group.keys():
        item = group[key]
        if isinstance(item, h5py.Group):
            list_datasets(item, prefix=f"{prefix}{key}/")
        else:
            print(f"  {prefix}{key}: shape={item.shape}, dtype={item.dtype}")


list_datasets(demo)

## Visualize Images (if available)


In [ ]:
# Check if images exist in the demo
if "obs" in demo and "image" in demo["obs"]:
    images = demo["obs"]["image"][:]
    print(f"Image shape: {images.shape}")

    # Display a grid of images from the trajectory
    num_frames = min(8, len(images))
    frame_indices = np.linspace(0, len(images) - 1, num_frames, dtype=int)

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()

    for _i, (ax, idx) in enumerate(zip(axes, frame_indices, strict=False)):
        img = images[idx]
        # Handle different image formats
        if img.dtype == np.uint8:
            ax.imshow(img)
        else:
            # Normalize float images
            img_normalized = (img - img.min()) / (img.max() - img.min() + 1e-8)
            ax.imshow(img_normalized)
        ax.set_title(f"Frame {idx}")
        ax.axis("off")

    plt.suptitle(f"Images from {demo_name}")
    plt.tight_layout()
    plt.show()
else:
    print("No images found in this demo.")

## Plot Observation Trajectories


In [ ]:
# Plot non-image observations
obs_group = demo["obs"]
obs_names = [k for k in obs_group.keys() if k != "image"]

if obs_names:
    num_obs = len(obs_names)
    fig, axes = plt.subplots(num_obs, 1, figsize=(12, 3 * num_obs), sharex=True)
    if num_obs == 1:
        axes = [axes]

    for ax, obs_name in zip(axes, obs_names, strict=False):
        obs_data = obs_group[obs_name][:]
        timesteps = np.arange(len(obs_data))

        # Plot each dimension
        if obs_data.ndim == 1:
            ax.plot(timesteps, obs_data, label=obs_name)
        else:
            for dim in range(min(obs_data.shape[1], 10)):  # Limit to 10 dimensions
                ax.plot(timesteps, obs_data[:, dim], label=f"dim_{dim}", alpha=0.7)

        ax.set_ylabel(obs_name)
        ax.legend(loc="upper right", fontsize=8, ncol=min(5, obs_data.shape[-1] if obs_data.ndim > 1 else 1))
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel("Timestep")
    plt.suptitle(f"Observations from {demo_name}")
    plt.tight_layout()
    plt.show()
else:
    print("No non-image observations found.")

## Plot Actions


In [ ]:
# Plot actions if available
if "actions" in demo:
    actions = demo["actions"][:]
    print(f"Actions shape: {actions.shape}")

    timesteps = np.arange(len(actions))

    fig, ax = plt.subplots(figsize=(12, 4))

    # Plot each action dimension
    num_dims = min(actions.shape[1], 20)  # Limit for readability
    for dim in range(num_dims):
        ax.plot(timesteps, actions[:, dim], label=f"dim_{dim}", alpha=0.7)

    ax.set_xlabel("Timestep")
    ax.set_ylabel("Action Value")
    ax.set_title(f"Actions from {demo_name}")
    ax.legend(loc="upper right", fontsize=7, ncol=5)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No actions found in this demo.")

## Cleanup


In [ ]:
# Close the HDF5 file when done
h5_file.close()
print("HDF5 file closed.")